# 1. INSTALAR LIBRERÍAS

In [ ]:
!pip install -q ultralytics roboflow

# 2. DESCARGAR EL DATASET (versión con las 3 clases)

In [ ]:
from roboflow import Roboflow
from google.colab import userdata

rf = Roboflow(api_key=userdata.get("ROBOFLOW_API_KEY"))   # clave guardada en Secretos de Colab
project = rf.workspace("david-santiago-calderon-giraldo").project("ecovision-pcmsb")
version = project.version(7)          # versión actual
dataset = version.download("yolov8")

# 3. ENTRENAR YOLOv8 NANO

In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")
results = model.train(
    data=f"{dataset.location}/data.yaml",
    epochs=60,
    patience=15,       # se detiene solo si deja de mejorar
    imgsz=640,
    batch=16,
    seed=0,            # reproducible (RNF-06)
    name="ecovision_3clases",
    plots=True,
)

run_dir = str(results.save_dir)        # carpeta real de ESTE entrenamiento
best_pt = f"{run_dir}/weights/best.pt"
print("Carpeta del entrenamiento:", run_dir)

# 4. EVALUAR CON EL CONJUNTO DE PRUEBA (Tablas 10 y 11) — ejecutar una sola vez

In [ ]:
best = YOLO(best_pt)
print("Clases del modelo:", best.names)   # debe mostrar las 3 clases

for size in (640, 320):
    r = best.val(data=f"{dataset.location}/data.yaml", split="test",
                 imgsz=size, plots=True, name=f"test_{size}")
    print(f"\n== imgsz {size} ==  P={r.box.mp:.3f}  R={r.box.mr:.3f}  "
          f"mAP50={r.box.map50:.3f}  mAP50-95={r.box.map:.3f}")
    for i, c in enumerate(r.box.ap_class_index):
        p, rc, ap50, ap = r.box.class_result(i)
        print(f"  {best.names[c]:<20} P={p:.3f} R={rc:.3f} mAP50={ap50:.3f} mAP50-95={ap:.3f}")

# 5. EXPORTAR Y DESCARGAR TODO

In [ ]:
best.export(format="onnx")   # opcional
!zip -qr /content/ecovision_resultados.zip runs/detect

from google.colab import files
files.download(best_pt)
files.download("/content/ecovision_resultados.zip")

# 6. CONTEO DE INSTANCIAS POR PARTICIÓN (Tabla 8)



In [ ]:
import glob, collections, yaml
names = yaml.safe_load(open(f"{dataset.location}/data.yaml"))["names"]
for split in ["train", "valid", "test"]:
    c, n = collections.Counter(), 0
    for f in glob.glob(f"{dataset.location}/{split}/labels/*.txt"):
        n += 1
        for line in open(f):
            if line.strip():
                c[int(line.split()[0])] += 1
    print(f"{split:<6} imágenes: {n:<5}", {names[k]: v for k, v in sorted(c.items())})

# 7. DATOS PARA LAS TABLAS 9 Y 12

In [ ]:
# A) Épocas y mejor época (Tabla 8)
import pandas as pd
df = pd.read_csv(f"{run_dir}/results.csv"); df.columns = df.columns.str.strip()
fit = 0.1*df["metrics/mAP50(B)"] + 0.9*df["metrics/mAP50-95(B)"]
b = fit.idxmax()
print("Épocas ejecutadas:", len(df), "| Mejor época:", int(df.loc[b, "epoch"]) + 1,
      "| mAP50 val:", round(df.loc[b, "metrics/mAP50(B)"], 3),
      "| mAP50-95 val:", round(df.loc[b, "metrics/mAP50-95(B)"], 3))

# B) Matriz de confusión a 640 px (Tabla 11)
r = best.val(data=f"{dataset.location}/data.yaml", split="test", imgsz=640, plots=False, verbose=False)
m = r.confusion_matrix.matrix.astype(int)
nombres = list(best.names.values()) + ["fondo"]
print("filas = predicho, columnas = real")
print(f"{'':<20}" + "".join(f"{n:>20}" for n in nombres))
for i, fila in enumerate(m):
    print(f"{nombres[i]:<20}" + "".join(f"{v:>20}" for v in fila))

# 8. MATRIZ DE CONFUSIÓN A 640 px (Tabla 12)

In [ ]:
r = best.val(data=f"{dataset.location}/data.yaml", split="test", imgsz=640,
             plots=True, verbose=False, name="cm_640")
m = r.confusion_matrix.matrix.astype(int)
nombres = list(best.names.values()) + ["fondo"]
print("filas = predicho, columnas = real")
print(f"{'':<20}" + "".join(f"{n:>20}" for n in nombres))
for i, fila in enumerate(m):
    print(f"{nombres[i]:<20}" + "".join(f"{v:>20}" for v in fila))